In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


For each model, calculate:

- mean QSS
- median QSS
- number / percentage of datasets with QSS>0
- an equal-asset-class-weighted mean QSS because the asset classes contain different numbers of series

In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised"
)

FREQUENCIES = {
    "Daily": ROOT / "output_daily",
    "Weekly": ROOT / "output_weekly",
}


# ============================================================
# BUILD QSS SUMMARY
# ============================================================

def build_qss_summary(
    base_dir,
    frequency,
):

    freq_lower = frequency.lower()

    qss = pd.read_csv(
        base_dir
        / "qss_summary_all_models.csv"
    )


    # ========================================================
    # DATASET-LEVEL INDICATORS
    # ========================================================

    qss["positive_qss"] = (
        qss["qss_0.1"] > 0
    )

    qss["negative_qss"] = (
        qss["qss_0.1"] < 0
    )

    qss["zero_qss"] = (
        qss["qss_0.1"] == 0
    )


    # ========================================================
    # ORDINARY EQUAL-SERIES SUMMARY
    #
    # Every dataset gets equal weight.
    # ========================================================

    series_summary = (
        qss
        .groupby(
            "model",
            as_index=False,
        )
        .agg(
            mean_qss=(
                "qss_0.1",
                "mean",
            ),
            median_qss=(
                "qss_0.1",
                "median",
            ),
            min_qss=(
                "qss_0.1",
                "min",
            ),
            max_qss=(
                "qss_0.1",
                "max",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
            n_positive_qss=(
                "positive_qss",
                "sum",
            ),
            n_negative_qss=(
                "negative_qss",
                "sum",
            ),
        )
    )


    series_summary["pct_positive_qss"] = (
        100
        * series_summary["n_positive_qss"]
        / series_summary["n_datasets"]
    )

    series_summary["pct_negative_qss"] = (
        100
        * series_summary["n_negative_qss"]
        / series_summary["n_datasets"]
    )


    # ========================================================
    # ASSET-CLASS MEANS
    #
    # First calculate one mean QSS per:
    # model × asset class.
    # ========================================================

    asset_means = (
        qss
        .groupby(
            [
                "model",
                "asset_class",
            ],
            as_index=False,
        )
        .agg(
            asset_mean_qss=(
                "qss_0.1",
                "mean",
            ),
            asset_median_qss=(
                "qss_0.1",
                "median",
            ),
            n_datasets=(
                "dataset",
                "count",
            ),
            n_positive_qss=(
                "positive_qss",
                "sum",
            ),
        )
    )


    asset_means["pct_positive_qss"] = (
        100
        * asset_means["n_positive_qss"]
        / asset_means["n_datasets"]
    )


    # ========================================================
    # EQUAL-ASSET-CLASS SUMMARY
    #
    # Each of the 8 asset classes contributes equally,
    # regardless of how many datasets it contains.
    # ========================================================

    equal_asset_summary = (
        asset_means
        .groupby(
            "model",
            as_index=False,
        )
        .agg(
            equal_asset_mean_qss=(
                "asset_mean_qss",
                "mean",
            ),
            equal_asset_median_qss=(
                "asset_median_qss",
                "mean",
            ),
            mean_asset_pct_positive=(
                "pct_positive_qss",
                "mean",
            ),
            n_asset_classes=(
                "asset_class",
                "count",
            ),
        )
    )


    # ========================================================
    # MERGE
    # ========================================================

    summary = series_summary.merge(
        equal_asset_summary,
        on="model",
        how="left",
        validate="one_to_one",
    )

    summary.insert(
        0,
        "frequency",
        frequency,
    )


    # ========================================================
    # SORT
    #
    # Higher QSS = better
    # ========================================================

    summary = (
        summary
        .sort_values(
            "mean_qss",
            ascending=False,
        )
        .reset_index(drop=True)
    )


    return (
        summary,
        asset_means,
    )


# ============================================================
# RUN DAILY + WEEKLY
# ============================================================

all_summaries = []

for frequency, base_dir in FREQUENCIES.items():

    summary, asset_means = (
        build_qss_summary(
            base_dir=base_dir,
            frequency=frequency,
        )
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    summary_path = (
        base_dir
        / f"qss_robustness_summary_{frequency.lower()}.csv"
    )

    asset_path = (
        base_dir
        / f"qss_asset_class_summary_{frequency.lower()}.csv"
    )


    summary.to_csv(
        summary_path,
        index=False,
    )

    asset_means.to_csv(
        asset_path,
        index=False,
    )


    # --------------------------------------------------------
    # Print compact version
    # --------------------------------------------------------

    print()
    print("=" * 110)
    print(
        f"{frequency.upper()} QSS ROBUSTNESS"
    )
    print("=" * 110)

    display_cols = [
        "model",
        "mean_qss",
        "median_qss",
        "n_positive_qss",
        "pct_positive_qss",
        "equal_asset_mean_qss",
        "mean_asset_pct_positive",
    ]

    print(
        summary[
            display_cols
        ]
        .round(4)
        .to_string(index=False)
    )


    print()
    print(
        f"Saved to: {summary_path}"
    )

    all_summaries.append(
        summary
    )


# ============================================================
# COMBINED DAILY + WEEKLY
# ============================================================

combined = pd.concat(
    all_summaries,
    ignore_index=True,
)

combined.to_csv(
    ROOT
    / "qss_robustness_summary_daily_weekly.csv",
    index=False,
)

# Loss-difference uncertainty

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20
TAU = 0.1

N_BOOT = 2000
BLOCK_LENGTHS = [10, 20, 40]
SEED = 42

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# DAILY
# ------------------------------------------------------------

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_daily"
)

DATA_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_log_return"
    r"\arima_split_datasets\output_daily"
)


MODEL_DIRS = {
    "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
}


MODEL_SUFFIXES = {
    "GARCH(1,1)": "_garch_1_1",
    "LightGBM": "_lightgbm",
    "Sundial": "_sundial",
    "Lag-Llama": "_lag-llama",
    "Chronos-T5": "_chronos-t5-small",
    "Toto": "_toto",
    "TimesFM": "_timesfm-2.5-200m",
    "Moirai": "_moirai-2.0-r-small",
    "Moirai-MoE": "_moirai-moe-1.0-r-base",
}


# ============================================================
# BASIC HELPERS
# ============================================================

def pinball_loss(y, q, tau=TAU):

    error = y - q

    return np.maximum(
        tau * error,
        (tau - 1.0) * error,
    )


def load_test_xy(test_path):

    df = pd.read_csv(test_path)

    x_cols = [
        f"X_{i}"
        for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    X = df[x_cols].to_numpy(dtype=float)
    Y = df[y_cols].to_numpy(dtype=float)

    return X, Y


def get_q10_index(levels):

    idx = np.argmin(
        np.abs(levels - TAU)
    )

    if not np.isclose(
        levels[idx],
        TAU,
    ):
        raise ValueError(
            f"No q{TAU} found: {levels}"
        )

    return idx


# ============================================================
# GARCH FILE HELPERS
# ============================================================

def clean_garch_dataset_name(dataset_name):

    return (
        dataset_name
        .lower()
        .replace(" historical data", "")
        .replace(
            "_no_preprocess_lookback-250_horizon-20_slide-1",
            "",
        )
        .replace(" ", "_")
        .replace(".", "")
    )


def find_garch_quantile_file(
    model_root,
    asset_class,
    dataset_name,
):

    asset_dir = (
        model_root
        / asset_class
    )

    clean_name = clean_garch_dataset_name(
        dataset_name
    )

    prefix = (
        f"{asset_class.lower()}_"
        f"{clean_name}_"
        f"garch_1_1_train_test"
    )

    expected = (
        asset_dir
        / f"{prefix}_quantiles.npz"
    )

    if expected.exists():
        return expected

    matches = list(
        asset_dir.glob(
            f"*{clean_name}*"
            f"garch_1_1_train_test*"
            f"quantiles.npz"
        )
    )

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not identify GARCH quantiles for "
        f"{dataset_name}; found {len(matches)}."
    )


def find_quantile_file(
    model_name,
    asset_class,
    dataset_name,
):

    model_root = MODEL_DIRS[
        model_name
    ]

    if model_name == "GARCH(1,1)":

        return find_garch_quantile_file(
            model_root,
            asset_class,
            dataset_name,
        )

    suffix = MODEL_SUFFIXES[
        model_name
    ]

    matches = list(
        (
            model_root
            / asset_class
        ).glob(
            f"test_{dataset_name}"
            f"{suffix}_quantiles.npz"
        )
    )

    if len(matches) != 1:
        raise FileNotFoundError(
            f"{model_name}, {dataset_name}: "
            f"found {len(matches)} quantile files."
        )

    return matches[0]


# ============================================================
# MOVING-BLOCK BOOTSTRAP
# ============================================================

def moving_block_bootstrap_mean_ci(
    values,
    block_length,
    n_boot=N_BOOT,
    seed=SEED,
):
    """
    Bootstrap confidence interval for the mean of a
    serially dependent sequence using overlapping
    moving blocks.

    values:
        one paired loss difference per forecast origin

    Positive mean:
        model loss < empirical-reference loss
    """

    values = np.asarray(
        values,
        dtype=float,
    )

    n = len(values)

    if block_length > n:
        raise ValueError(
            f"Block length {block_length} > n={n}"
        )

    rng = np.random.default_rng(
        seed
    )

    # All possible contiguous block starts
    max_start = (
        n - block_length
    )

    n_blocks_needed = int(
        np.ceil(
            n / block_length
        )
    )

    boot_means = np.empty(
        n_boot,
        dtype=float,
    )

    for b in range(n_boot):

        starts = rng.integers(
            0,
            max_start + 1,
            size=n_blocks_needed,
        )

        sample = np.concatenate([
            values[
                s:s + block_length
            ]
            for s in starts
        ])

        # Trim to original length
        sample = sample[:n]

        boot_means[b] = np.mean(
            sample
        )

    estimate = float(
        np.mean(values)
    )

    ci_lower = float(
        np.quantile(
            boot_means,
            0.025,
        )
    )

    ci_upper = float(
        np.quantile(
            boot_means,
            0.975,
        )
    )

    return {
        "mean_loss_difference": estimate,
        "ci_lower_95": ci_lower,
        "ci_upper_95": ci_upper,
        "significantly_positive": (
            ci_lower > 0
        ),
        "significantly_negative": (
            ci_upper < 0
        ),
    }


# ============================================================
# RUN
# ============================================================

rows = []


for model_name in MODEL_DIRS:

    print()
    print("=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    for asset_class, asset_name in ASSET_ORDER.items():

        test_dir = (
            DATA_DIR
            / asset_class
        )

        test_files = sorted(
            test_dir.glob(
                "test_*.csv"
            )
        )

        for test_path in test_files:

            try:

                dataset_name = (
                    test_path.stem
                    .removeprefix("test_")
                )

                X, Y = load_test_xy(
                    test_path
                )

                # --------------------------------------------
                # Empirical q0.1 reference
                # --------------------------------------------

                q_ref = np.quantile(
                    X,
                    TAU,
                    axis=1,
                )

                q_ref = q_ref[:, None]

                reference_loss = pinball_loss(
                    Y,
                    q_ref,
                    TAU,
                )

                # --------------------------------------------
                # Model q0.1
                # --------------------------------------------

                quantile_path = (
                    find_quantile_file(
                        model_name,
                        asset_class,
                        dataset_name,
                    )
                )

                with np.load(
                    quantile_path
                ) as q_file:

                    Q = np.asarray(
                        q_file["quantiles"],
                        dtype=float,
                    )

                    levels = np.asarray(
                        q_file[
                            "quantile_levels"
                        ],
                        dtype=float,
                    )

                q10_idx = get_q10_index(
                    levels
                )

                model_q10 = Q[
                    :,
                    :,
                    q10_idx,
                ]

                if model_q10.shape != Y.shape:
                    raise ValueError(
                        f"{model_name} {dataset_name}: "
                        f"q10 {model_q10.shape}, "
                        f"Y {Y.shape}"
                    )

                model_loss = pinball_loss(
                    Y,
                    model_q10,
                    TAU,
                )

                # --------------------------------------------
                # One paired loss difference per origin
                #
                # Average the 20 horizons within each origin.
                #
                # Overall mean of this vector is exactly the
                # aggregate mean loss difference.
                # --------------------------------------------

                loss_difference_by_origin = (
                    np.mean(
                        reference_loss
                        - model_loss,
                        axis=1,
                    )
                )

                # --------------------------------------------
                # Bootstrap several block lengths
                # --------------------------------------------

                for block_length in BLOCK_LENGTHS:

                    result = (
                        moving_block_bootstrap_mean_ci(
                            loss_difference_by_origin,
                            block_length=block_length,
                        )
                    )

                    rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        "block_length": block_length,
                        "n_origins": len(
                            loss_difference_by_origin
                        ),
                        **result,
                    })

                print(
                    f"  {dataset_name}: done"
                )

            except Exception as exc:

                print(
                    f"  ERROR {test_path.name}: "
                    f"{exc}"
                )


# ============================================================
# SAVE DATASET-LEVEL RESULTS
# ============================================================

results = pd.DataFrame(
    rows
)

results.to_csv(
    BASE_DIR
    / "qss_loss_difference_block_bootstrap_daily.csv",
    index=False,
)


# ============================================================
# SUMMARY FOR PRIMARY BLOCK LENGTH = 20
# ============================================================

primary = results[
    results["block_length"] == 20
].copy()


model_summary = (
    primary
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        mean_loss_difference=(
            "mean_loss_difference",
            "mean",
        ),
        median_loss_difference=(
            "mean_loss_difference",
            "median",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
        n_ci_positive=(
            "significantly_positive",
            "sum",
        ),
        n_ci_negative=(
            "significantly_negative",
            "sum",
        ),
    )
)


model_summary["pct_ci_positive"] = (
    100
    * model_summary["n_ci_positive"]
    / model_summary["n_datasets"]
)

model_summary["pct_ci_negative"] = (
    100
    * model_summary["n_ci_negative"]
    / model_summary["n_datasets"]
)


model_summary = model_summary.sort_values(
    "mean_loss_difference",
    ascending=False,
).reset_index(drop=True)


model_summary.to_csv(
    BASE_DIR
    / "qss_loss_difference_bootstrap_summary_daily.csv",
    index=False,
)


# ============================================================
# BLOCK-LENGTH SENSITIVITY
# ============================================================

sensitivity = (
    results
    .groupby(
        [
            "model",
            "block_length",
        ],
        as_index=False,
    )
    .agg(
        mean_loss_difference=(
            "mean_loss_difference",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
        n_ci_positive=(
            "significantly_positive",
            "sum",
        ),
        n_ci_negative=(
            "significantly_negative",
            "sum",
        ),
    )
)


sensitivity["pct_ci_positive"] = (
    100
    * sensitivity["n_ci_positive"]
    / sensitivity["n_datasets"]
)

sensitivity["pct_ci_negative"] = (
    100
    * sensitivity["n_ci_negative"]
    / sensitivity["n_datasets"]
)


sensitivity.to_csv(
    BASE_DIR
    / "qss_loss_difference_block_sensitivity_daily.csv",
    index=False,
)


# ============================================================
# PRINT
# ============================================================

print()
print("=" * 100)
print("BLOCK BOOTSTRAP LOSS-DIFFERENCE SUMMARY — DAILY")
print("=" * 100)

print(
    model_summary
    .round(6)
    .to_string(index=False)
)

print()
print("=" * 100)
print("BLOCK-LENGTH SENSITIVITY")
print("=" * 100)

print(
    sensitivity
    .round(4)
    .to_string(index=False)
)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20
TAU = 0.1

N_BOOT = 2000
BLOCK_LENGTHS = [10, 20, 40]
SEED = 42

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# WEEKLY
# ------------------------------------------------------------

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_weekly"
)

DATA_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_log_return"
    r"\arima_split_datasets\output_weekly"
)


MODEL_DIRS = {
    "GARCH(1,1)": BASE_DIR / "garch_standardized_train_test",
    "LightGBM": BASE_DIR / "lightgbm_standardized_train_test",
    "Sundial": BASE_DIR / "sundial_standardized_train_test",
    "Lag-Llama": BASE_DIR / "lag-llama_standardized_train_test",
    "Chronos-T5": BASE_DIR / "chronos_standardized_train_test",
    "Toto": BASE_DIR / "toto_standardized_train_test",
    "TimesFM": BASE_DIR / "timesfm_standardized_train_test",
    "Moirai": BASE_DIR / "moirai_standardized_train_test",
    "Moirai-MoE": BASE_DIR / "moirai_moe_standardized_train_test",
}


MODEL_SUFFIXES = {
    "GARCH(1,1)": "_garch_1_1",
    "LightGBM": "_lightgbm",
    "Sundial": "_sundial",
    "Lag-Llama": "_lag-llama",
    "Chronos-T5": "_chronos-t5-small",
    "Toto": "_toto",
    "TimesFM": "_timesfm-2.5-200m",
    "Moirai": "_moirai-2.0-r-small",
    "Moirai-MoE": "_moirai-moe-1.0-r-base",
}


# ============================================================
# BASIC HELPERS
# ============================================================

def pinball_loss(y, q, tau=TAU):

    error = y - q

    return np.maximum(
        tau * error,
        (tau - 1.0) * error,
    )


def load_test_xy(test_path):

    df = pd.read_csv(test_path)

    x_cols = [
        f"X_{i}"
        for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    X = df[x_cols].to_numpy(dtype=float)
    Y = df[y_cols].to_numpy(dtype=float)

    return X, Y


def get_q10_index(levels):

    idx = np.argmin(
        np.abs(levels - TAU)
    )

    if not np.isclose(
        levels[idx],
        TAU,
    ):
        raise ValueError(
            f"No q{TAU} found: {levels}"
        )

    return idx


# ============================================================
# GARCH FILE HELPERS
# ============================================================

def clean_garch_dataset_name(dataset_name):

    return (
        dataset_name
        .lower()
        .replace(" historical data", "")
        .replace(
            "_no_preprocess_lookback-250_horizon-20_slide-1",
            "",
        )
        .replace(" ", "_")
        .replace(".", "")
    )


def find_garch_quantile_file(
    model_root,
    asset_class,
    dataset_name,
):

    asset_dir = (
        model_root
        / asset_class
    )

    clean_name = clean_garch_dataset_name(
        dataset_name
    )

    prefix = (
        f"{asset_class.lower()}_"
        f"{clean_name}_"
        f"garch_1_1_train_test"
    )

    expected = (
        asset_dir
        / f"{prefix}_quantiles.npz"
    )

    if expected.exists():
        return expected

    matches = list(
        asset_dir.glob(
            f"*{clean_name}*"
            f"garch_1_1_train_test*"
            f"quantiles.npz"
        )
    )

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not identify GARCH quantiles for "
        f"{dataset_name}; found {len(matches)}."
    )


def find_quantile_file(
    model_name,
    asset_class,
    dataset_name,
):

    model_root = MODEL_DIRS[
        model_name
    ]

    if model_name == "GARCH(1,1)":

        return find_garch_quantile_file(
            model_root,
            asset_class,
            dataset_name,
        )

    suffix = MODEL_SUFFIXES[
        model_name
    ]

    matches = list(
        (
            model_root
            / asset_class
        ).glob(
            f"test_{dataset_name}"
            f"{suffix}_quantiles.npz"
        )
    )

    if len(matches) != 1:
        raise FileNotFoundError(
            f"{model_name}, {dataset_name}: "
            f"found {len(matches)} quantile files."
        )

    return matches[0]


# ============================================================
# MOVING-BLOCK BOOTSTRAP
# ============================================================

def moving_block_bootstrap_mean_ci(
    values,
    block_length,
    n_boot=N_BOOT,
    seed=SEED,
):
    """
    Bootstrap confidence interval for the mean of a
    serially dependent sequence using overlapping
    moving blocks.

    values:
        one paired loss difference per forecast origin

    Positive mean:
        model loss < empirical-reference loss
    """

    values = np.asarray(
        values,
        dtype=float,
    )

    n = len(values)

    if block_length > n:
        raise ValueError(
            f"Block length {block_length} > n={n}"
        )

    rng = np.random.default_rng(
        seed
    )

    # All possible contiguous block starts
    max_start = (
        n - block_length
    )

    n_blocks_needed = int(
        np.ceil(
            n / block_length
        )
    )

    boot_means = np.empty(
        n_boot,
        dtype=float,
    )

    for b in range(n_boot):

        starts = rng.integers(
            0,
            max_start + 1,
            size=n_blocks_needed,
        )

        sample = np.concatenate([
            values[
                s:s + block_length
            ]
            for s in starts
        ])

        # Trim to original length
        sample = sample[:n]

        boot_means[b] = np.mean(
            sample
        )

    estimate = float(
        np.mean(values)
    )

    ci_lower = float(
        np.quantile(
            boot_means,
            0.025,
        )
    )

    ci_upper = float(
        np.quantile(
            boot_means,
            0.975,
        )
    )

    return {
        "mean_loss_difference": estimate,
        "ci_lower_95": ci_lower,
        "ci_upper_95": ci_upper,
        "significantly_positive": (
            ci_lower > 0
        ),
        "significantly_negative": (
            ci_upper < 0
        ),
    }


# ============================================================
# RUN
# ============================================================

rows = []


for model_name in MODEL_DIRS:

    print()
    print("=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    for asset_class, asset_name in ASSET_ORDER.items():

        test_dir = (
            DATA_DIR
            / asset_class
        )

        test_files = sorted(
            test_dir.glob(
                "test_*.csv"
            )
        )

        for test_path in test_files:

            try:

                dataset_name = (
                    test_path.stem
                    .removeprefix("test_")
                )

                X, Y = load_test_xy(
                    test_path
                )

                # --------------------------------------------
                # Empirical q0.1 reference
                # --------------------------------------------

                q_ref = np.quantile(
                    X,
                    TAU,
                    axis=1,
                )

                q_ref = q_ref[:, None]

                reference_loss = pinball_loss(
                    Y,
                    q_ref,
                    TAU,
                )

                # --------------------------------------------
                # Model q0.1
                # --------------------------------------------

                quantile_path = (
                    find_quantile_file(
                        model_name,
                        asset_class,
                        dataset_name,
                    )
                )

                with np.load(
                    quantile_path
                ) as q_file:

                    Q = np.asarray(
                        q_file["quantiles"],
                        dtype=float,
                    )

                    levels = np.asarray(
                        q_file[
                            "quantile_levels"
                        ],
                        dtype=float,
                    )

                q10_idx = get_q10_index(
                    levels
                )

                model_q10 = Q[
                    :,
                    :,
                    q10_idx,
                ]

                if model_q10.shape != Y.shape:
                    raise ValueError(
                        f"{model_name} {dataset_name}: "
                        f"q10 {model_q10.shape}, "
                        f"Y {Y.shape}"
                    )

                model_loss = pinball_loss(
                    Y,
                    model_q10,
                    TAU,
                )

                # --------------------------------------------
                # One paired loss difference per origin
                #
                # Average the 20 horizons within each origin.
                #
                # Overall mean of this vector is exactly the
                # aggregate mean loss difference.
                # --------------------------------------------

                loss_difference_by_origin = (
                    np.mean(
                        reference_loss
                        - model_loss,
                        axis=1,
                    )
                )

                # --------------------------------------------
                # Bootstrap several block lengths
                # --------------------------------------------

                for block_length in BLOCK_LENGTHS:

                    result = (
                        moving_block_bootstrap_mean_ci(
                            loss_difference_by_origin,
                            block_length=block_length,
                        )
                    )

                    rows.append({
                        "model": model_name,
                        "asset_class": asset_name,
                        "dataset": dataset_name,
                        "block_length": block_length,
                        "n_origins": len(
                            loss_difference_by_origin
                        ),
                        **result,
                    })

                print(
                    f"  {dataset_name}: done"
                )

            except Exception as exc:

                print(
                    f"  ERROR {test_path.name}: "
                    f"{exc}"
                )


# ============================================================
# SAVE DATASET-LEVEL RESULTS
# ============================================================

results = pd.DataFrame(
    rows
)

results.to_csv(
    BASE_DIR
    / "qss_loss_difference_block_bootstrap_weekly.csv",
    index=False,
)


# ============================================================
# SUMMARY FOR PRIMARY BLOCK LENGTH = 20
# ============================================================

primary = results[
    results["block_length"] == 20
].copy()


model_summary = (
    primary
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        mean_loss_difference=(
            "mean_loss_difference",
            "mean",
        ),
        median_loss_difference=(
            "mean_loss_difference",
            "median",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
        n_ci_positive=(
            "significantly_positive",
            "sum",
        ),
        n_ci_negative=(
            "significantly_negative",
            "sum",
        ),
    )
)


model_summary["pct_ci_positive"] = (
    100
    * model_summary["n_ci_positive"]
    / model_summary["n_datasets"]
)

model_summary["pct_ci_negative"] = (
    100
    * model_summary["n_ci_negative"]
    / model_summary["n_datasets"]
)


model_summary = model_summary.sort_values(
    "mean_loss_difference",
    ascending=False,
).reset_index(drop=True)


model_summary.to_csv(
    BASE_DIR
    / "qss_loss_difference_bootstrap_summary_weekly.csv",
    index=False,
)


# ============================================================
# BLOCK-LENGTH SENSITIVITY
# ============================================================

sensitivity = (
    results
    .groupby(
        [
            "model",
            "block_length",
        ],
        as_index=False,
    )
    .agg(
        mean_loss_difference=(
            "mean_loss_difference",
            "mean",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
        n_ci_positive=(
            "significantly_positive",
            "sum",
        ),
        n_ci_negative=(
            "significantly_negative",
            "sum",
        ),
    )
)


sensitivity["pct_ci_positive"] = (
    100
    * sensitivity["n_ci_positive"]
    / sensitivity["n_datasets"]
)

sensitivity["pct_ci_negative"] = (
    100
    * sensitivity["n_ci_negative"]
    / sensitivity["n_datasets"]
)


sensitivity.to_csv(
    BASE_DIR
    / "qss_loss_difference_block_sensitivity_weekly.csv",
    index=False,
)


# ============================================================
# PRINT
# ============================================================

print()
print("=" * 100)
print("BLOCK BOOTSTRAP LOSS-DIFFERENCE SUMMARY — WEEKLY")
print("=" * 100)

print(
    model_summary
    .round(6)
    .to_string(index=False)
)

print()
print("=" * 100)
print("BLOCK-LENGTH SENSITIVITY")
print("=" * 100)

print(
    sensitivity
    .round(4)
    .to_string(index=False)
)